# Notebook 23 — E6: Order-robust intrusion detection validation (train-only)

**Objective:** Investigate whether serialized five-record context improves rare-attack detection beyond a current-record-only LightGBM baseline on an **alternative group-disjoint split made entirely from the official UNSW-NB15 TRAINING CSV**. Investigate nonoverlapping windows and shuffled-context controls. Preserve already completed official-test results; **NEVER access or tune using the official testing-set CSV or its stored predictions.**

## Exactly preregistered tests

| Candidate | Input to LightGBM | Training windows | Evaluation |
| --- | --- | --- | --- |
| `target_full` | 42 fields from newest target row only | All overlapping windows | Full validation and disjoint subset |
| `intact_full` | Original 5 × 42 fields + 234 statistics (444 total) | All overlapping windows | Same validation rows |
| `shuffled_full` | 444 fields but all 4 historical rows moved *together* to a random target, with six statistics recalculated | All overlapping windows | Same validation targets, separately permuted in each split |
| `target_nonoverlap` | 42 newest-row fields | Every fifth window (disjoint train rows) | Every fifth validation window |
| `intact_nonoverlap` | 444 window fields | Exactly same every-fifth-window training subset as target_nonoverlap | Same every-fifth validation window |

All models use the same **10-class sqrt class balancing capped at 12**, frozen **58 boosting rounds**, **63 leaves**, **learning rate 0.06**, **seed 42**, and no validation-controlled early stopping or architecture choice. Alternative split seed **2026** is fixed before fitting. Identical 42-feature rows are grouped to keep such duplicates from crossing development training/validation. The validation subset uses disjoint (nonoverlapping) five-record windows, although it is NOT proof of independent real network sessions or chronological time.

E5 found that same-class adjacency in the training CSV is 56.88% while chance under independent class margins is 20.98%, with a 47,911-record longest same-class run. **This is suspicious class clustering, not proof of traffic chronology.** This E6 experiment is a robustness *diagnostic*, not itself a novel temporal model.

## Reproducibility and automated preservation

- Source SHA-256 and 42-feature manifest validated. Only original `UNSW_NB15_training-set.csv` read; no test.
- Every completed fitted booster and all per-row validation predictions saved with SHA256 checks, and immediately uploaded to a separate **unpublished PRIVATE draft GitHub release**. Aggregate confusion, class reports, source feature importances, group split hashes, and matched comparison statistics committed publicly.
- If Kaggle crashes later, re-running this notebook restores and verifies previously uploaded complete candidates **without retraining them**.
- No model's training round count or hyperparameters are chosen using E6 validation. No official test predictions or samples are opened.

## Kaggle setup

**Accelerator None (CPU)**, Internet **ON**, dataset **`dsshayan/unsw-nb15`**, existing secret **`pushRare`** enabled, **Save Version → Save & Run All**. No original E2/E3 models need restoration because this is a fresh matched split experiment. Five CPU fits and immediate private releases may take several minutes. No GPU required.

**Final log:** `E6 ORDER-ROBUSTNESS COMPLETE — 5 PRIVATE MODEL BACKUPS + PUBLIC AGGREGATES`.

If E6's contextual benefit disappears on nonoverlapping windows or against fair shuffled controls, the paper should not claim robust spatiotemporal learning. If it persists, the **next independent step** is validation on a dataset with authentic timestamps/flow identifiers, followed by a meaningful methodological contribution.

In [ ]:
# ONE CELL — 5 fixed CPU fits, alternative TRAIN-only grouped split, no official test.
import os,sys,hashlib,subprocess
from pathlib import Path
from kaggle_secrets import UserSecretsClient

REPO=Path("/kaggle/working/rare-attack-ids-e6")
OUT=Path("/kaggle/working/research_outputs/E6-ORDER-ROBUSTNESS-001")
TRAIN=Path("/kaggle/input/datasets/dsshayan/unsw-nb15/UNSW_NB15_training-set.csv")
assert TRAIN.is_file(), "Attach original UNSW-NB15 Kaggle TRAINING dataset"
with TRAIN.open("rb") as f:
    assert hashlib.file_digest(f,"sha256").hexdigest()==(
        "bec7dd5ec88dc2a0ccc7a07879d338395ed7421750f675fd0339e07dfe0648fa"
    ),"Original official TRAIN source SHA256 mismatch"
token=UserSecretsClient().get_secret("pushRare")
assert token, "Authorize existing pushRare GitHub secret for private model backups"
if REPO.is_dir():
    subprocess.run(["git","-C",str(REPO),"pull","--ff-only"],check=True)
else:
    subprocess.run(["git","clone","--depth","1",
        "https://github.com/itsCodeBakery/rare-attack-spatiotemporal-ids.git",
        str(REPO)],check=True)
env=os.environ.copy()
env["GITHUB_TOKEN"]=token
env["PYTHONPATH"]=str(REPO/"src")
env["OMP_NUM_THREADS"]="4"
env["OPENBLAS_NUM_THREADS"]="1"
sha=subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],
                            text=True).strip()
print("E6 SOURCE COMMIT:",sha,flush=True)
print("OFFICIAL TEST WILL NOT BE OPENED OR REUSED.",flush=True)

subprocess.run([
    sys.executable,str(REPO/"scripts/run_e6_order_robustness.py"),
    "--repo",str(REPO),
    "--config",str(REPO/"configs/e6_order_robustness_001.json"),
    "--train-csv",str(TRAIN),
    "--output",str(OUT),"--auto-archive"
],cwd=str(REPO),env=env,check=True)

print("\nE6 COMPLETE MODELS:",flush=True)
print((OUT/"comparison_all_models.csv").read_text(),flush=True)
print("E6 MATCHED CONTEXT COMPARISONS:",flush=True)
print((OUT/"paired_comparisons.csv").read_text(),flush=True)
print("Private candidate checkpoints and public aggregate findings backed up.",flush=True)
